# Libraries

In [22]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from google.cloud import storage
from sklearn.preprocessing import StandardScaler
from google.cloud import bigquery
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sklearn.preprocessing import MinMaxScaler

In [23]:
'''
This code just loads the survival_table.csv file and assigns a test / training split.
I moved this into its own file so I could load the same test dataset without
having to retrain the model.
'''
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sksurv.util import Surv
from sklearn.model_selection import train_test_split

def load_data():
  df = pd.read_csv("survival_table.csv")

  df['srvc_dt'] = pd.to_datetime(df['srvc_dt'])
  df['next_fill_date'] = pd.to_datetime(df['next_fill_date'])

  df = df.dropna(subset=['next_fill_date']).copy()

  df.loc[:,'time_to_non_adherence'] = df['days_since_last_fill'].astype(float)
  df.loc[:,'event'] = df['non_adherence_event'].astype(bool)

  #Aggregate by beneficiary 
  df_agg = df.groupby("bene_id").agg({
    "time_to_non_adherence": "mean",
    "event":"max",
    "qty_dspnsd_num":"sum",
    "beneficiary_age_at_service": "mean",
    "zip_cd": "first",
    "drug_cvrg_status_cat":"first",
    "pde_id":"count",
    "days_since_last_fill":"std",
    "claim_start_year":"max"
  }).reset_index()

  #Calculating MPR
  df_agg["med_adherence_percent"]=(df_agg["qty_dspnsd_num"]/(df_agg["time_to_non_adherence"]+1))*100
  df_agg["med_adherence_percent"] = df_agg["med_adherence_percent"].clip(0,100)
  
  df_model = df_agg

  # Keep the normal status

  df_copy = df_agg.copy()
  # print(df_copy["zip_cd"])

  categorical_columns = ['drug_cvrg_status_cat']
  label_encoders = {}

  for col in categorical_columns:
    le = LabelEncoder()
    df_model[col]=le.fit_transform(df_model[col])
    label_encoders[col]= le

  #Model building
  y = Surv.from_arrays(event=df_model["event"],time=df_model["time_to_non_adherence"])
  X = df_model.drop(columns =["time_to_non_adherence","event","pde_id","bene_id","claim_start_year"])

  bene_series = df_model['bene_id']
  # Test/train split
  X_temp, X_test, y_temp, y_test, bene_temp, bene_test = train_test_split(X,y,bene_series, test_size = 0.2, random_state = 42)
  X_train, X_val, y_train, y_val, bene_train, bene_val = train_test_split(X_temp, y_temp, bene_temp, test_size = 0.2, random_state= 42)
  
  return df_copy, X, y, bene_series, X_train, X_val, X_test, y_train, y_val, y_test, bene_train, bene_val, bene_test

def concordance_scorer(estimator,X,y):
    pred_risk = estimator.predict(X)
    event = y["event"]
    time = y["time"]
    c_index = concordance_index_censored(event, time, pred_risk)[0]
    return c_index

In [24]:
df, X, y, bene_series, X_train, X_val, X_test, y_train, y_val, y_test, bene_train, bene_val, bene_test   = load_data()

print(X_train["zip_cd"])

# print(df["beneficiary_age_at_service"])

458     90731.0
3694    60804.0
1549    17402.0
261     94964.0
3273    21144.0
         ...   
1649    19446.0
1283    78550.0
3635        NaN
412     92688.0
2376    11369.0
Name: zip_cd, Length: 3150, dtype: float64


# Train a simple model

In [25]:
# Don't scale inputs
# scaler = StandardScaler()
# scaler.fit(X_train)  # fit on train only

# # Transform train/val/test
# X_train_scaled = scaler.transform(X_train)
# X_val_scaled   = scaler.transform(X_val)
# X_test_scaled  = scaler.transform(X_test)

# Find best params
X_train_inner, X_val_inner, y_train_inner, y_val_inner = train_test_split(X_train, y_train, test_size = 0.2, random_state=42)
param_grid = {"n_estimators": [100,150,200],"min_samples_split":[5,10],
              "max_depth": [5,8,10]}
model = RandomSurvivalForest(random_state=42, n_jobs=-1) 
grid_search = GridSearchCV(model, param_grid, cv=5,
                           scoring=concordance_scorer,n_jobs=-1) #verbose=2,
print("Starting grid search")
grid_search.fit(X_train_inner,y_train_inner)

#Best parameters
best_params = grid_search.best_params_
print("Best Hyperparamters:", best_params)
print(f"Best Concordance Index: {grid_search.best_score_:.3f}")
model = RandomSurvivalForest(**best_params, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

Starting grid search


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Hyperparamters: {'max_depth': 8, 'min_samples_split': 10, 'n_estimators': 200}
Best Concordance Index: 0.823


RandomSurvivalForest(max_depth=8, min_samples_split=10, n_estimators=200,
                     n_jobs=-1, random_state=42)

In [ ]:
# Make Predictions
min_max_scaler = MinMaxScaler()
min_max_scaler.fit(model.predict(X_test).reshape(-1, 1))
y_pred_train = min_max_scaler.transform(model.predict(X_train).reshape(-1, 1))
y_pred_val   = min_max_scaler.transform(model.predict(X_val).reshape(-1, 1))
y_pred_test  = min_max_scaler.transform(model.predict(X_test).reshape(-1, 1))

In [27]:
#Upload min/Max scaler

# GCS Path
project_id = "spatial-earth-449020-m3"
bucket_name = "capstone-group-data"

# Initialize client
client = storage.Client(project=project_id)
bucket = client.bucket(bucket_name)

model_bytes = pickle.dumps(min_max_scaler)

model_path = "final_models/medication_adherence_min_max_scaler.pkl"

# Upload model
blob_model = bucket.blob(model_path)
blob_model.upload_from_string(
    model_bytes,
    content_type="application/octet-stream"
)

# Upload Models

In [28]:
# GCS Path
project_id = "spatial-earth-449020-m3"
bucket_name = "capstone-group-data"

model_path  = "final_models/medication_adherence_rsf_unscaled.pkl"
# Griffin use: "final_models/readmissions_rf.pkl"
# Jessica use: "final_models/medication_adherence_rsf.pkl"
# Lexi use: "final_models/claim_cost_rgr.pkl"

feat_path   = "final_models/medication_adherence_rsf.json"
# Griffin use: "final_models/readmissions_feat_order.json"
# Jessica use: "final_models/medication_adherence_rsf.pkl"
# Lexi use: "final_models/claim_cost_rgr.pkl"

# Initialize client
client = storage.Client(project=project_id)
bucket = client.bucket(bucket_name)

# Derive feature order
feature_order = X.columns.tolist()

# Serialize model into memory
model_bytes = pickle.dumps(model)
feature_json = json.dumps(feature_order)

# Upload model
blob_model = bucket.blob(model_path)
blob_model.upload_from_string(
    model_bytes,
    content_type="application/octet-stream"
)

#Upload Feature Order
blob_feat = bucket.blob(feat_path)
blob_feat.upload_from_string(
    feature_json,
    content_type="application/json"
)

print("Files successfully uploaded to GCS")

Files successfully uploaded to GCS


#Upload Data to Big Query

In [29]:
# Copy scaled data
X_train_unscaled = X_train.copy()
X_val_unscaled   = X_val.copy()
X_test_unscaled  = X_test.copy()

# Identified numeric columns
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

# # If you standardardized your data, unstardardized
# X_train_unscaled[numeric_cols] = scaler.inverse_transform(X_train_unscaled[numeric_cols])
# X_val_unscaled[numeric_cols]   = scaler.inverse_transform(X_val_unscaled[numeric_cols])
# X_test_unscaled[numeric_cols]  = scaler.inverse_transform(X_test_unscaled[numeric_cols])

# Reattach IDs
X_train_unscaled['bene_id'] = bene_train.reset_index(drop=True)
X_val_unscaled['bene_id'] = bene_val.reset_index(drop=True) #Delete if you didn't use validation
X_test_unscaled['bene_id'] = bene_test.reset_index(drop=True)

# Attach Actuals
# X_train_unscaled['actual'] = y_train.reset_index(drop=True)
# X_val_unscaled['actual'] = y_val.reset_index(drop=True) #Delete if you didn't use validation
# X_test_unscaled['actual'] = y_test.reset_index(drop=True)

# Attach Predictions
X_train_unscaled['predicted'] = y_pred_train
X_val_unscaled['predicted']   = y_pred_val
X_test_unscaled['predicted']  = y_pred_test

# Union datasets together
df_all_unscaled = pd.concat(
    [X_train_unscaled, X_val_unscaled, X_test_unscaled],
    ignore_index=True
)

# Convert any dates to integers
#df_all_unscaled['benefit_yr'] = df_all_unscaled['benefit_yr'].astype(int)

# Round off any very small decimals generated from inverse scaling
df_all_unscaled = df_all_unscaled.round(6)

# Force very small values to 0 or 1
for col in df_all_unscaled.select_dtypes(include=[float]).columns:
    unique_vals = df_all_unscaled[col].dropna().unique()
    if unique_vals.min() >= -1e-6 and unique_vals.max() <= 1 + 1e-6:
        df_all_unscaled[col] = np.where(
            df_all_unscaled[col] >= 0.5, 1, 0
        ).astype(int)


client = bigquery.Client(project="spatial-earth-449020-m3")
table_id = (
    "spatial-earth-449020-m3."
    "capstone_finalized_model_datasets_w_predictions."
    "beneficiary_medication_adherence_w_predictions_rsf" #See below
)

# For table name
# Griffin use: "beneficiary_inpatient_readmission_w_predictions_rf"
# Jessica use: "beneficiary_medication_adherence_w_predictions_rsf"
# Lexi use: "beneficiary_claim_costs_w_predictions_regression"


# Configure the load job: overwrite the existing table, autodetect schema
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
    create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED
)

# Load the DataFrame into BigQuery
load_job = client.load_table_from_dataframe(
    df_all_unscaled,
    table_id,
    job_config=job_config
)

# 6) Wait for it to complete
load_job.result()

print(
    f"Data successfully loaded into: {table_id}\n"
    "All existing data was replaced with new rows."
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Data successfully loaded into: spatial-earth-449020-m3.capstone_finalized_model_datasets_w_predictions.beneficiary_medication_adherence_w_predictions_rsf
All existing data was replaced with new rows.
